<style>
:root {
  color-scheme: dark;
  --lyric-yellow: #ffff64;
  --lyric-black: #111111;
  --lyric-white: #f8f8f8;
  --bs-body-bg: var(--lyric-black);
  --bs-body-color: var(--lyric-white);
  --bs-border-color: #414141;
  --bs-secondary-color: #cccccc;
}
body { color: var(--lyric-white); background: var(--lyric-black); }
#title-block-header {
  background: var(--lyric-black);
  color: #fff;
  border-left: .5rem solid var(--lyric-yellow);
  border-bottom-color: var(--bs-border-color);
  padding: clamp(1rem, 5vw, 2.5rem);
  margin-bottom: 1.5rem;
}
#title-block-header .title { color: var(--lyric-yellow); font-weight: 750; letter-spacing: -.045em; line-height: 1.03; }
#title-block-header .subtitle { color: #fff; max-width: 42rem; margin-bottom: 0; }
main h2 { color: var(--lyric-white); border-bottom: 0; border-left: .35rem solid var(--lyric-yellow); padding-left: .55rem; font-weight: 720; letter-spacing: -.025em; }
main a { color: var(--lyric-yellow); text-underline-offset: .15em; }
#TOC a { color: var(--lyric-white); }
main a:hover, #TOC a:hover { background: var(--lyric-yellow); color: var(--lyric-black); }
main a:focus-visible, #TOC a:focus-visible { outline: 2px solid var(--lyric-yellow); outline-offset: 3px; }
#TOC .nav-link.active { color: var(--lyric-yellow); border-left-color: var(--lyric-yellow); font-weight: 650; }
main img { max-width: 100%; height: auto; }
figcaption { color: #cccccc; line-height: 1.45; }
</style>

How much wording comes back in a song? These charts compare English lyrics from 2010–2019 with 300–349 cleaned words.

In [ ]:
import csv
import io
import re
from contextlib import contextmanager
from pathlib import Path
from statistics import median, quantiles
from zipfile import ZipFile
from zlib import crc32

import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter

csv_path = Path("data/song_lyrics.csv")
zip_path = Path("data/song_lyrics.zip")
tags = {"pop": "Pop", "rap": "Rap", "rock": "Rock", "rb": "R&B", "country": "Country"}
section_tag = re.compile(r"\[[^\]\r\n]*\]")
word = re.compile(r"[a-z]+(?:'[a-z]+)*")
fields = ("tag", "year", "lyrics", "id", "language_cld3", "language_ft", "language")


@contextmanager
def open_lyrics():
    if csv_path.is_file():
        with csv_path.open(encoding="utf-8-sig", newline="") as source:
            yield source
    elif zip_path.is_file():
        with ZipFile(zip_path) as archive:
            with io.TextIOWrapper(
                archive.open("song_lyrics.csv"), encoding="utf-8-sig", newline=""
            ) as source:
                yield source
    else:
        raise FileNotFoundError("Place song_lyrics.zip or song_lyrics.csv in data/. See README.md.")


word_group_sizes = (2, 4, 8, 16, 32, 64)


def repeated_share(items):
    return 100 * (len(items) - len(set(items))) / len(items)


def word_group_repeat_share(words, size):
    groups = list(zip(*(words[offset:] for offset in range(size))))
    return repeated_share(groups)


csv.field_size_limit(20_000_000)
source_count = 0
seen_ids = set()
records = []
with open_lyrics() as source:
    rows = csv.reader(source)
    header = next(rows)
    tag_i, year_i, lyric_i, id_i, cld_i, ft_i, lang_i = (header.index(field) for field in fields)
    for row in rows:
        source_count += 1
        song_id = row[id_i]
        if crc32(song_id.encode("utf-8")) % 50:
            continue
        tag, year = row[tag_i], row[year_i]
        if tag not in tags or not (year.isdecimal() and 2010 <= int(year) <= 2019):
            continue
        if row[cld_i] != "en" or row[ft_i] != "en" or row[lang_i] != "en":
            continue
        if not song_id or song_id in seen_ids:
            continue
        seen_ids.add(song_id)

        cleaned = section_tag.sub("", row[lyric_i]).lower().replace("’", "'")
        lines = [tuple(word.findall(line)) for line in cleaned.splitlines()]
        lines = [line for line in lines if line]
        words = [token for line in lines for token in line]
        if not 250 <= len(words) <= 399:
            continue
        line_share = repeated_share(lines)
        group_shares = {size: word_group_repeat_share(words, size) for size in word_group_sizes}
        records.append((tag, len(words), line_share, group_shares))


def summarize(min_words, max_words):
    summary = {}
    for tag in tags:
        group = [record for record in records if record[0] == tag and min_words <= record[1] <= max_words]
        if group:
            summary[tag] = {
                "songs": len(group),
                "repeat": median(record[2] for record in group),
                "line_shares": [record[2] for record in group],
                "has_32": 100 * sum(record[3][32] > 0 for record in group) / len(group),
                "word_groups": {
                    size: median(record[3][size] for record in group)
                    for size in word_group_sizes
                },
            }
    return summary


main = summarize(300, 349)
wider = summarize(250, 399)
if not {"pop", "rap"} <= main.keys():
    raise ValueError("The input has no qualifying Pop or Rap rows. Check the dataset version.")

## How much of a lyric repeats?

Count each occurrence of a normalized line after its first appearance. In **A B A B C D**, two of the six written lines repeat.

In [ ]:
#| output: asis

def middle_half(tag):
    q1, _, q3 = quantiles(main[tag]["line_shares"], method="inclusive")
    return f"{q1:.1f}%–{q3:.1f}%"

pop, rap = main["pop"], main["rap"]
display(Markdown(
    f"The median repeated-line share is **{pop['repeat']:.1f}% for Pop** and "
    f"**{rap['repeat']:.1f}% for Rap**. Their middle halves overlap slightly: "
    f"{middle_half('pop')} for Pop and {middle_half('rap')} for Rap."
))

In [ ]:
#| label: fig-repeat-distribution
#| fig-cap: "Boxes span the 25th to 75th percentiles, dark marks show medians, and whiskers span the 10th to 90th percentiles."
#| fig-alt: "Horizontal box plots showing within-genre distributions of repeated-line shares for five Genius tags. Pop and Rap have only a small overlap between their middle halves."

yellow, ink, gray = "#ffff64", "#171717", "#6c6c6c"
paper, grid = "#ffffff", "#dedede"
colors = {"pop": yellow, "rap": ink}
plt.rcParams.update({
    "axes.facecolor": paper,
    "axes.edgecolor": gray,
    "axes.labelcolor": ink,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.facecolor": paper,
    "font.family": "DejaVu Sans",
    "text.color": ink,
    "xtick.color": ink,
    "ytick.color": ink,
})
ordered = sorted(main, key=lambda tag: main[tag]["repeat"], reverse=True)
labels = [f"{tags[tag]} ({main[tag]['songs']:,})" for tag in ordered]

fig, ax = plt.subplots(figsize=(7.5, 3.7), layout="constrained")
boxes = ax.boxplot([main[tag]["line_shares"] for tag in ordered], orientation="horizontal",
                   tick_labels=labels, widths=0.58, whis=(10, 90),
                   showfliers=False, patch_artist=True,
                   medianprops={"color": ink, "linewidth": 2},
                   whiskerprops={"color": gray}, capprops={"color": gray})
for box, tag in zip(boxes["boxes"], ordered):
    box.set(facecolor=colors[tag] if tag == "pop" else "#d4d4d4", edgecolor=ink)
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel="Share of written lines that repeat")
ax.xaxis.set_major_formatter(PercentFormatter(100, decimals=0))
ax.set_axisbelow(True)
ax.grid(axis="x", color=grid)
plt.show()

## Does the pattern hold for longer word groups?

Slide a window through the whole lyric, one word at a time. In *hold on hold on*, one of three two-word windows repeats. Windows can overlap and cross line breaks.

In [ ]:
#| output: asis

def pair_at(summary, size):
    pop_share = summary["pop"]["word_groups"][size]
    rap_share = summary["rap"]["word_groups"][size]
    return f"**{pop_share:.1f}% for Pop** and **{rap_share:.1f}% for Rap**"

display(Markdown(
    f"The median repeated-window share is {pair_at(main, 8)} at eight words, "
    f"{pair_at(main, 32)} at 32 words, and {pair_at(main, 64)} at 64 words. "
    f"In the wider 250–399-word sample, the 32-word medians are {pair_at(wider, 32)}. "
    f"Even when a median is close to zero, some songs have long repeats. "
    f"At least one exact 32-word sequence repeats in **{main['pop']['has_32']:.1f}%** "
    f"of Pop lyrics and **{main['rap']['has_32']:.1f}%** of Rap lyrics."
))

In [ ]:
#| label: fig-word-group-repeats
#| fig-cap: "Median per-lyric share of exact repeated word windows. Each series uses the same lyrics at all six lengths. Values are percentages of windows for that length."
#| fig-alt: "Line chart of repeated word window shares at six lengths from two to 64 words. Pop and Rap are highlighted against three other Genius tags."

fig, ax = plt.subplots(figsize=(7.5, 3.7), layout="constrained")
for tag in ordered:
    if tag not in colors:
        ax.plot(word_group_sizes, [main[tag]["word_groups"][size] for size in word_group_sizes],
                color=gray, alpha=0.48, linewidth=1.2, marker="o", markersize=4)
for tag, marker in (("pop", "o"), ("rap", "D")):
    values = [main[tag]["word_groups"][size] for size in word_group_sizes]
    line, = ax.plot(word_group_sizes, values, color=colors[tag], linewidth=2.2, marker=marker,
                    markeredgecolor=ink, linestyle="--" if tag == "rap" else "-",
                    label=tags[tag], zorder=3)
    if tag == "pop":
        line.set_path_effects([path_effects.withStroke(linewidth=3.6, foreground=ink)])
ax.set_xscale("log", base=2)
ax.set(xticks=word_group_sizes, xticklabels=[str(size) for size in word_group_sizes],
       xlabel="Words per window (doubles at each step)", ylabel="Median repeated windows",
       ylim=(0, 55))
ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
ax.legend(title="Gray lines: Country, Rock, R&B", frameon=False)
ax.set_axisbelow(True)
ax.grid(axis="y", color=grid)
plt.show()

## Data and method

The [Genius archive, version 1](https://www.kaggle.com/datasets/carlosgdcj/genius-song-lyrics-with-language-information/data) contributes about one in 50 song IDs via a fixed sampling rule. The analysis keeps one row per sampled ID, keeps rows marked English in all three language fields, removes bracketed annotations, and normalizes case and punctuation.

These are exact text matches. A single refrain can generate many overlapping word windows, and a refrain written once with a repeat instruction can be missed.

In [ ]:
#| output: asis

display(Markdown(
    f"The archive has {source_count:,} rows. The charts use "
    f"**{sum(group['songs'] for group in main.values()):,} lyrics** across five tags."
))